### Exclusion ZONE_SYMBOLOGY -> 1, 2

In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np

# Load the full National RZLT dataset (replace with your exact local file path)
rzlt_path = "Datasets/RZLT-Dataset.geojson"
gdf_rzlt = gpd.read_file(rzlt_path)

# Standardize Coordinate Reference System to Irish Transverse Mercator (EPSG:2157)
if gdf_rzlt.crs != "EPSG:2157":
    gdf_rzlt = gdf_rzlt.to_crs(epsg=2157)

print(f"Loaded {len(gdf_rzlt):,} total RZLT parcels.")
print(f"CRS: {gdf_rzlt.crs}")

Loaded 260,463 total RZLT parcels.
CRS: EPSG:2157


In [3]:
# Map ZONE_SYMBOLOGY codes to readable statutory definitions
symbology_map = {
    1: "Retained_Residential",
    2: "Retained_MixedUse",
    3: "Exclusion_Residential",
    4: "Exclusion_MixedUse",
    5: "Inclusion_Residential",
    6: "Inclusion_MixedUse"
}

gdf_rzlt['SYMBOLOGY_LABEL'] = gdf_rzlt['ZONE_SYMBOLOGY'].map(symbology_map)

# Filter to active inclusion/exclusion decisions (Codes 3, 4, 5, 6 -> 3,742 rows)
gdf_decisions = gdf_rzlt[gdf_rzlt['ZONE_SYMBOLOGY'].isin([3, 4, 5, 6])].copy()

# Create binary classification target:
# 1 = Proposed Inclusion (5, 6) | 0 = Proposed Exclusion (3, 4)
gdf_decisions['target'] = gdf_decisions['ZONE_SYMBOLOGY'].isin([5, 6]).astype(int)
gdf_decisions['DECISION_STATUS'] = np.where(gdf_decisions['target'] == 1, 'Inclusion', 'Exclusion')

print("=== Active Decision Subset Breakdown ===")
print(gdf_decisions['SYMBOLOGY_LABEL'].value_counts())
print("\n=== Binary Target Breakdown ===")
print(gdf_decisions['DECISION_STATUS'].value_counts())

=== Active Decision Subset Breakdown ===
SYMBOLOGY_LABEL
Inclusion_Residential    2907
Exclusion_Residential     312
Exclusion_MixedUse        276
Inclusion_MixedUse        247
Name: count, dtype: int64

=== Binary Target Breakdown ===
DECISION_STATUS
Inclusion    3154
Exclusion     588
Name: count, dtype: int64


In [4]:
# 1. Recalculate SITE_AREA in hectares where missing or zero (m² / 10,000)
computed_area_ha = gdf_decisions.geometry.area / 10000.0
gdf_decisions['SITE_AREA'] = gdf_decisions['SITE_AREA'].fillna(computed_area_ha)
gdf_decisions.loc[gdf_decisions['SITE_AREA'] <= 0, 'SITE_AREA'] = computed_area_ha

# 2. Clean categorical predictors
gdf_decisions['ZONE_GZT'] = gdf_decisions['ZONE_GZT'].fillna("Unknown").astype(str).str.strip()
gdf_decisions['LOCAL_AUTHORITY'] = gdf_decisions['LOCAL_AUTHORITY'].fillna("Unknown").astype(str).str.strip()

# 3. Extract EPSG:2157 centroids in meters
gdf_decisions['centroid_x'] = gdf_decisions.geometry.centroid.x
gdf_decisions['centroid_y'] = gdf_decisions.geometry.centroid.y

# 4. Drop tiny geometry slivers (< 0.005 hectares / 50 m²)
gdf_decisions = gdf_decisions[gdf_decisions['SITE_AREA'] >= 0.005].copy()

print(f"Cleaned decision dataset rows: {len(gdf_decisions):,}")

Cleaned decision dataset rows: 3,665


In [5]:
# Select core modeling & metadata columns (excluding ZONE_SYMBOLOGY to prevent leakage later)
keep_cols = [
    'PARCEL_ID',
    'LOCAL_AUTHORITY',
    'ZONE_GZT',
    'ZONE_ORIG',
    'DATE_ADDED',
    'SITE_AREA',
    'DECISION_STATUS',
    'target',
    'centroid_x',
    'centroid_y',
    'geometry'
]
available_cols = [c for c in keep_cols if c in gdf_decisions.columns]
gdf_clean = gdf_decisions[available_cols].copy()

# Option A: 1:1 Balanced Sample (588 Exclusions + 588 Sampled Inclusions = 1,176 rows)
df_excl = gdf_clean[gdf_clean['target'] == 0]
df_incl = gdf_clean[gdf_clean['target'] == 1].sample(n=len(df_excl), random_state=42)

gdf_balanced_sample = pd.concat([df_excl, df_incl], ignore_index=True)

# Option B: Full Active Decisions Dataset (~3,742 rows, ~5.3:1 ratio)
gdf_full_decisions = gdf_clean.reset_index(drop=True)

print(f"Option A (1:1 Balanced Sample): {len(gdf_balanced_sample):,} rows")
print(gdf_balanced_sample['DECISION_STATUS'].value_counts())

print(f"\nOption B (Full Active Decisions): {len(gdf_full_decisions):,} rows")
print(gdf_full_decisions['DECISION_STATUS'].value_counts())

Option A (1:1 Balanced Sample): 1,074 rows
DECISION_STATUS
Exclusion    537
Inclusion    537
Name: count, dtype: int64

Option B (Full Active Decisions): 3,665 rows
DECISION_STATUS
Inclusion    3128
Exclusion     537
Name: count, dtype: int64


In [6]:
output_dir = "Datasets/RZLT_Samples"
os.makedirs(output_dir, exist_ok=True)

balanced_path = os.path.join(output_dir, "RZLT_Balanced_Sample_1to1.geojson")
full_path = os.path.join(output_dir, "RZLT_Active_Decisions_Full.geojson")

gdf_balanced_sample.to_file(balanced_path, driver="GeoJSON")
gdf_full_decisions.to_file(full_path, driver="GeoJSON")

print(f"Saved 1:1 balanced sample  -> {balanced_path}")
print(f"Saved full active decisions -> {full_path}")

Saved 1:1 balanced sample  -> Datasets/RZLT_Samples\RZLT_Balanced_Sample_1to1.geojson
Saved full active decisions -> Datasets/RZLT_Samples\RZLT_Active_Decisions_Full.geojson
